# TP-MCTS Experiments

Two experiment scripts, run from this notebook.

| Script | Purpose |
|--------|---------|
| `run_mcts_heuristic_comparison.py` | TP-MCTS score across 4 NASA scenarios × 5 heuristics |
| `run_heuristic_runtime_per_call.py` | Per-call timing (wrapper + worker + cache hit/miss) |

**Setup:** clone the repo and `cd` into it first (same as `demo.ipynb` cells 1-4).

In [ ]:
# Clone repo (skip if already done in this Colab session)
import os

if not os.path.exists('/content/tp_mcts'):
    %cd /content
    !git clone https://github.com/eliezerRevach/tp_mcts.git

%cd /content/tp_mcts
!pip -q install dill numpy pandas
print('Ready:', os.getcwd())

## Config

Edit the values below, then run the cells for each experiment.

In [ ]:
# ── Shared config ─────────────────────────────────────────────────────────

RUNS                 = 20     # runs per (scenario × heuristic)
SEED                 = 123    # random seed — same seed across all runs
SEARCH_TIME          = 1      # MCTS search time per step (seconds)
EXPLORATION_CONSTANT = 10.0   # UCT exploration constant C
REWARD_MODE          = "deadline"  # "deadline" or "terminal"

# Scenario grid — Script 1 only
OBJECTS      = [2, 3]      # object_amount values
DEADLINES    = [25, 35]    # deadline values

# Heuristics — used by both scripts
# Options: ptrpg_old  baseline  baseline_cached  atomic_exact  atomic_exact_cached
HEURISTICS   = ["ptrpg_old", "baseline", "baseline_cached", "atomic_exact", "atomic_exact_cached"]

# Runtime benchmark settings — Script 2 only
RT_OBJECTS   = 2
RT_DEADLINE  = 25
RT_H_DEPTH   = 25
RT_MAX_STEPS = 90

# Output paths (created automatically inside the repo)
MCTS_CSV    = "results/mcts_heuristic_comparison.csv"
RUNTIME_CSV = "results/heuristic_runtime_per_call.csv"

print("Config:")
print(f"  MCTS scenarios : objects={OBJECTS}  deadlines={DEADLINES}  runs={RUNS}  seed={SEED}  C={EXPLORATION_CONSTANT}")
print(f"  Heuristics     : {HEURISTICS}")
print(f"  Runtime bench  : nasa_rover obj={RT_OBJECTS}  deadline={RT_DEADLINE}  depth={RT_H_DEPTH}")

## Script 1 — MCTS Heuristic Comparison

Runs TP-MCTS on each **(object_amount, deadline) × heuristic** combination.

- Results are saved **incrementally** — partial data survives a Colab timeout.
- Output: `results/mcts_heuristic_comparison.csv`

In [ ]:
import subprocess

_h   = " ".join(HEURISTICS)
_obj = " ".join(str(o) for o in OBJECTS)
_dl  = " ".join(str(d) for d in DEADLINES)

cmd = [
    "python", "scripts/run_mcts_heuristic_comparison.py",
    "--runs",                 str(RUNS),
    "--seed",                 str(SEED),
    "--search_time",          str(SEARCH_TIME),
    "--exploration_constant", str(EXPLORATION_CONSTANT),
    "--reward_mode",          REWARD_MODE,
    "--output",               MCTS_CSV,
    "--objects",              *[str(o) for o in OBJECTS],
    "--deadlines",            *[str(d) for d in DEADLINES],
    "--heuristics",           *HEURISTICS,
]

print("Running:", " ".join(cmd), flush=True)
print()

# Stream output live so you see per-experiment progress
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

In [ ]:
import pandas as pd

df = pd.read_csv(MCTS_CSV)

# Pivot: rows = (objects, deadline), columns = heuristic, values = success_rate
pivot = df.pivot_table(
    index=["object_amount", "deadline"],
    columns="heuristic",
    values="success_rate",
    aggfunc="first",
)
print("=== Success rate by scenario × heuristic ===")
print(pivot.to_string())

print("\n=== Full results table ===")
cols = ["domain", "object_amount", "deadline", "heuristic",
        "amount_success", "success_rate", "avg_success_time", "std_success_time"]
print(df[cols].to_string(index=False))

## Script 2 — Heuristic Per-Call Runtime Benchmark

Runs `greedy_parallel` on one scenario for each heuristic and measures:

- `wrapper_avg_call_sec` — total heuristic call cost (includes STN work)
- `worker_avg_call_sec` — pure propagation cost
- `worker_cache_hit_avg_sec` / `worker_cache_miss_avg_sec` — cache breakdown

Output: `results/heuristic_runtime_per_call.csv`

In [ ]:
import subprocess

cmd = [
    "python", "scripts/run_heuristic_runtime_per_call.py",
    "--domain",         "nasa_rover",
    "--object_amount",  str(RT_OBJECTS),
    "--deadline",       str(RT_DEADLINE),
    "--heuristic_depth",str(RT_H_DEPTH),
    "--max_steps",      str(RT_MAX_STEPS),
    "--seed",           str(SEED),
    "--output",         RUNTIME_CSV,
    "--heuristics",     *HEURISTICS,
]

print("Running:", " ".join(cmd), flush=True)
print()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

In [ ]:
import pandas as pd

df_rt = pd.read_csv(RUNTIME_CSV)

timing_cols = [
    "heuristic",
    "wrapper_avg_call_sec",
    "worker_avg_call_sec",
    "worker_cache_hit_avg_sec",
    "worker_cache_miss_avg_sec",
    "worker_cache_hits",
    "worker_cache_misses",
    "plan_success",
]

df_sorted = df_rt[timing_cols].sort_values("wrapper_avg_call_sec")

print("=== Per-call runtime ranking (fastest → slowest) ===")
print(df_sorted.to_string(index=False))